# PocketAcc – fine-tune Qwen2-VL-2B on Kaggle (free GPU)

**Before running:** Settings (right panel) → **Accelerator: GPU T4 x2** (or P100) and **Internet: On**.
Run the cells top to bottom. Total time ≈ 3–5 hours (most of it is fine-tuning and evaluation).
Everything is saved under `/kaggle/working/pocketacc` – download `pocketacc_results.zip` at the end.

In [ ]:
!nvidia-smi

In [ ]:
REPO = "https://github.com/anshace07/pocketacc.git"   # <- your GitHub username
!git clone -q $REPO /kaggle/working/pocketacc
%cd /kaggle/working/pocketacc
!apt-get -qq install -y poppler-utils tesseract-ocr > /dev/null
!pip -q install -r requirements.txt -r requirements-gpu.txt

## Step 1–3: build the page-level dataset (≈20 min on CPU, deterministic seeds)

In [ ]:
!python src/build_dataset.py --out data/vlm

In [ ]:
!python scripts/smoke_test.py data/vlm   # tiny-model test of the training code (1–2 min)

## Step 4a: will it fit? (peak GPU memory for inference and one training step)

In [ ]:
!python scripts/vram_test.py --data data/vlm

## Step 4b: baseline – the model BEFORE fine-tuning (zero-shot) and the traditional OCR baseline

In [ ]:
!python src/evaluate.py predict --system vlm --no-4bit --split test_unseen --limit 24 --out results/base_unseen.jsonl
!python src/evaluate.py predict --system vlm --no-4bit --split test_seen   --limit 24 --out results/base_seen.jsonl
!python src/evaluate.py predict --system tesseract --split test_unseen --limit 24 --out results/tess_unseen.jsonl
!python src/evaluate.py predict --system tesseract --split test_seen   --limit 24 --out results/tess_seen.jsonl

## Step 4c: QLoRA fine-tuning (rank 8, 1 epoch)

In [ ]:
!python src/finetune.py --data data/vlm --out runs/lora_r8 --rank 8 --epochs 1 --grad-accum 8

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv("runs/lora_r8/training_log.csv")
ax = log.plot(x="step", y="train_loss", figsize=(7,3.5)); log.dropna(subset=["val_loss"]).plot(x="step", y="val_loss", ax=ax, marker="o")
plt.ylabel("loss"); plt.title("QLoRA fine-tuning"); plt.savefig("results/loss_curve.png", dpi=150, bbox_inches="tight"); plt.show()

## Step 4d: evaluate the fine-tuned model

In [ ]:
!python src/evaluate.py predict --system vlm --no-4bit --adapter runs/lora_r8/best_adapter --split test_unseen --limit 24 --out results/ft_unseen.jsonl
!python src/evaluate.py predict --system vlm --no-4bit --adapter runs/lora_r8/best_adapter --split test_seen   --limit 24 --out results/ft_seen.jsonl

In [ ]:
for name in ["tess", "base", "ft"]:
    for split in ["seen", "unseen"]:
        print(f"\n===== {name} / test_{split} =====")
        !python src/evaluate.py score results/{name}_{split}.jsonl --split test_{split} --json-out results/score_{name}_{split}.json

In [ ]:
!zip -qr /kaggle/working/pocketacc_results.zip results runs/lora_r8/best_adapter runs/lora_r8/training_log.csv runs/lora_r8/run_config.json && ls -lh /kaggle/working/pocketacc_results.zip